# Semi-synthetic correlation-stress experiment

Semi-synthetic correlation-stress experiment.

Real SUPPORT2 covariates are kept so that the natural correlation structure is
preserved. The outcome is generated from a known logistic model of five "true"
features (with one interaction). For each true feature a proxy is added whose
correlation with it is rho. Explanations are scored on how much of their
attribution lands on the true features rather than on proxies or on real but
irrelevant covariates.

## Setup
Loads `00_config` (paths) and the library notebooks from this folder.

In [7]:
# Load other notebooks in this folder as modules (all notebooks live in one folder).
import os, sys, types
import nbformat

NB_DIR = os.getcwd()   # open Jupyter in the notebooks folder


def import_notebook(name, filename):
    """Execute the code cells of a notebook in this folder into a module named `name`."""
    if name in sys.modules:
        return sys.modules[name]
    path = os.path.join(NB_DIR, filename)
    mod = types.ModuleType(name)
    mod.__file__ = path
    sys.modules[name] = mod
    for cell in nbformat.read(path, as_version=4).cells:
        if cell.cell_type == "code":
            exec(compile(cell.source, path, "exec"), mod.__dict__)
    return mod


config = import_notebook("config", "00_config.ipynb")

In [8]:
data = import_notebook("clinxai.data", "01_data_loaders.ipynb")
core = import_notebook("clinxai.core", "02_core_methods.ipynb")

In [9]:
import os
import sys
import time
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
OUT = config.RESULTS_DIR
TRUE = ["age", "crea", "alb", "meanbp", "resp"]
BETA = np.array([1.0, 0.8, -0.8, -0.6, 0.6])
GAMMA = 0.5  # age x crea interaction
RHOS = [0.0, 0.5, 0.7, 0.9, 0.95, 0.99]
SEEDS = [0, 1]
MODELS = ["LR", "XGB", "MLP", "EBM"]
N_EXPLAIN = 50

### `build` (function)

In [10]:
def build(rho, seed):
    d = data.load("SUPPORT2")
    num = [c for c in d.X.columns if c not in d.cat and c not in ("sps", "aps")]
    X = d.X[num].astype(float)
    X = X.fillna(X.median())  # UCI SUPPORT2 keeps missing values; the R copy had none
    X = (X - X.mean()) / X.std()
    rng = np.random.default_rng(seed)
    Zt = X[TRUE].values
    logit = Zt @ BETA + GAMMA * Zt[:, 0] * Zt[:, 1]
    b = np.quantile(logit, 0.65)  # about 35% prevalence
    y = rng.binomial(1, 1 / (1 + np.exp(-(logit - b))))
    if rho > 0:
        for j, c in enumerate(TRUE):
            X[f"{c}_proxy"] = rho * X[c] + np.sqrt(1 - rho ** 2) * rng.normal(size=len(X))
    return X, y

### `run` (function)

In [11]:
def run():
    rows = []
    os.makedirs(os.path.join(OUT, "semisynthetic"), exist_ok=True)
    for rho in RHOS:
        for seed in SEEDS:
            X, y = build(rho, seed)
            cols = list(X.columns)
            true_idx = [cols.index(c) for c in TRUE]
            proxy_idx = [i for i, c in enumerate(cols) if c.endswith("_proxy")]
            tr, te = train_test_split(np.arange(len(y)), test_size=0.3, stratify=y, random_state=seed)
            enc = core.Encoder().fit(X.iloc[tr], [])
            Ztr, Zte = enc.transform(X.iloc[tr]), enc.transform(X.iloc[te])
            rng = np.random.default_rng(seed)
            Zbg = Ztr[rng.choice(len(Ztr), 50, replace=False)]
            xs, _ = train_test_split(np.arange(len(te)), train_size=N_EXPLAIN, stratify=y[te], random_state=seed)
            Zx = Zte[xs]
            for kind in MODELS:
                t = time.time()
                params = core.tune(kind, enc, Ztr, y[tr], seed=seed, n_iter=8)
                m = core.make_model(kind, enc, seed, params).fit(Ztr, y[tr])
                auc = roc_auc_score(y[te], m.predict_proba(Zte)[:, 1])
                res, _ = core.explain_all(kind, m, enc, Ztr, Zbg, Zx, seed=seed)
                for a, A in res.items():
                    absA = np.abs(A)
                    top5 = core.topk_sets(A, 5)
                    local_p5 = np.mean([len(set(r) & set(true_idx)) / 5 for r in top5])
                    g = absA.mean(axis=0)
                    glob_p5 = len(set(np.argsort(-g)[:5]) & set(true_idx)) / 5
                    share_true = (absA[:, true_idx].sum(1) / absA.sum(1)).mean()
                    share_proxy = (absA[:, proxy_idx].sum(1) / absA.sum(1)).mean() if proxy_idx else 0.0
                    rows.append(dict(rho=rho, seed=seed, model=kind, explainer=a, auroc=auc, local_p5=local_p5,
                                     global_p5=glob_p5, share_true=share_true, share_proxy=share_proxy,
                                     n_features=len(cols)))
                print(f"rho={rho} seed={seed} {kind} auc={auc:.3f} {time.time() - t:.0f}s", flush=True)
            pd.DataFrame(rows).to_csv(os.path.join(OUT, "semisynthetic", "semisynthetic.csv"), index=False)

## Run
About 1 hour. Output: `OUTPUT_DIR/results/semisynthetic/semisynthetic.csv`.

In [12]:
run()

[SUPPORT2] reading D:\New_Research_2027\Clinical_XAI_Faithfulness\uci_downloads\uci_880_support2.csv
rho=0.0 seed=0 LR auc=0.831 4s
rho=0.0 seed=0 XGB auc=0.831 10s
rho=0.0 seed=0 MLP auc=0.828 28s
rho=0.0 seed=0 EBM auc=0.826 9s
[SUPPORT2] reading D:\New_Research_2027\Clinical_XAI_Faithfulness\uci_downloads\uci_880_support2.csv
rho=0.0 seed=1 LR auc=0.818 4s
rho=0.0 seed=1 XGB auc=0.820 8s
rho=0.0 seed=1 MLP auc=0.811 25s
rho=0.0 seed=1 EBM auc=0.814 11s
[SUPPORT2] reading D:\New_Research_2027\Clinical_XAI_Faithfulness\uci_downloads\uci_880_support2.csv
rho=0.5 seed=0 LR auc=0.831 4s
rho=0.5 seed=0 XGB auc=0.830 11s
rho=0.5 seed=0 MLP auc=0.826 29s
rho=0.5 seed=0 EBM auc=0.825 10s
[SUPPORT2] reading D:\New_Research_2027\Clinical_XAI_Faithfulness\uci_downloads\uci_880_support2.csv
rho=0.5 seed=1 LR auc=0.818 4s
rho=0.5 seed=1 XGB auc=0.820 10s
rho=0.5 seed=1 MLP auc=0.805 28s
rho=0.5 seed=1 EBM auc=0.813 13s
[SUPPORT2] reading D:\New_Research_2027\Clinical_XAI_Faithfulness\uci_download